# Sentiment Analysis: Classifying Text as Positive, Neutral or Negative
**Oasis Infobyte SIP · Data Analytics · Level 1 · Task 4**

**Objective:** build machine learning models that classify the sentiment of text (positive, negative or neutral), and use the results to understand customer opinion.

**Dataset:** `sentiment_reviews.csv` - 3,135 short product reviews / tweets with a `sentiment` label (columns: id, text, sentiment).

> The file included here is generated by `generate_data.py` (tweet-style product feedback with mentions, URLs, hashtags, negation, mixed opinions and a few mislabelled rows). To use a real dataset such as Kaggle's *Twitter Sentiment Analysis*, *Amazon Product Reviews* or *IMDB*, save it as `sentiment_reviews.csv` with columns `text` and `sentiment`, re-run, and update the written observations.
>
> **Running in Google Colab:** NLTK and WordCloud are pre-installed there. Locally, run `pip install -r requirements.txt`. If NLTK is not available the notebook falls back to scikit-learn's stopword list, and if WordCloud is missing it shows top-word bar charts instead. Numbers in the observations come from the fallback setup, so yours may differ by about a percentage point.

**Plan:** load and inspect -> class distribution -> text preprocessing -> train/test split -> TF-IDF -> train 3 models -> evaluate -> word clouds -> error analysis -> conclusion.

---
## 1. Load the data and inspect it

In [ ]:
import os, re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
os.makedirs("images", exist_ok=True)

df = pd.read_csv("sentiment_reviews.csv")
print("Shape:", df.shape)
print("Null values:", df.isnull().sum().to_dict(), "| duplicate texts:", df["text"].duplicated().sum())
df.sample(8, random_state=3)

**Observation:** the dataset has 3,135 texts with no missing values and no duplicate texts. The texts are informal: they contain @mentions, URLs, hashtags, capital letters and punctuation, so they need cleaning before modelling.

## 2. Class distribution (positive / neutral / negative)

In [ ]:
order = ["positive", "neutral", "negative"]
counts = df["sentiment"].value_counts().reindex(order)
pct = (counts / counts.sum() * 100).round(1)
print(pd.DataFrame({"count": counts, "percent": pct}))

colors = ["#16A34A", "#6B7280", "#DC2626"]
plt.figure(figsize=(7, 4.5))
plt.bar(counts.index, counts.values, color=colors)
for i, v in enumerate(counts.values):
    plt.text(i, v, f"{v}\n({pct.iloc[i]}%)", ha="center", va="bottom")
plt.title("Sentiment Distribution in the Dataset"); plt.ylabel("Number of texts"); plt.ylim(0, counts.max() * 1.18)
plt.savefig("images/01_sentiment_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

**Observation:** **positive 1,226 (39.1%)**, **negative 1,038 (33.1%)** and **neutral 871 (27.8%)**. The classes are moderately imbalanced but none is tiny, so a stratified split (next sections) keeps the same proportions in train and test, and we report macro-averaged precision, recall and F1 in addition to accuracy.

## 3. Text preprocessing pipeline

In [ ]:
# Stopwords: NLTK if available (Colab has it), otherwise scikit-learn's list as a fallback
try:
    import nltk
    nltk.download("stopwords", quiet=True)
    from nltk.corpus import stopwords
    STOP = set(stopwords.words("english"))
    STOP_SOURCE = "NLTK"
except Exception:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    STOP = set(ENGLISH_STOP_WORDS)
    STOP_SOURCE = "scikit-learn (fallback)"

# IMPORTANT: negation words flip sentiment ("not good"), so we must NOT remove them
STOP -= {"not", "no", "nor", "never", "cannot"}
USE_STEMMING = False      # optional: set True to apply NLTK PorterStemmer (needs nltk)
print("Stopword list used:", STOP_SOURCE, "| words:", len(STOP))

def preprocess(text):
    text = text.lower()                                   # 1. lowercase
    text = re.sub(r"http\S+|www\S+", " ", text)           # 2. remove URLs
    text = re.sub(r"@\w+", " ", text)                     # 3. remove @mentions
    text = text.replace("#", " ")                         # 4. keep hashtag word, drop '#'
    text = re.sub(r"'", "", text)                         # 5. don't -> dont
    text = re.sub(r"[^a-z\s]", " ", text)                 # 6. remove punctuation, digits, emojis
    tokens = text.split()                                 # 7. tokenisation
    tokens = [t for t in tokens if t not in STOP]         # 8. stopword removal
    if USE_STEMMING:                                      # 9. optional stemming
        from nltk.stem import PorterStemmer
        ps = PorterStemmer(); tokens = [ps.stem(t) for t in tokens]
    return " ".join(tokens)

df["clean_text"] = df["text"].apply(preprocess)
for i in range(4):
    print(f"ORIGINAL: {df['text'].iloc[i]}\nCLEANED : {df['clean_text'].iloc[i]}\n")

**Pipeline steps:** lowercase -> remove URLs -> remove @mentions -> drop `#` (keep the hashtag word) -> remove punctuation, digits and emoticons -> tokenise -> remove stopwords -> (optional stemming).

**Important decision:** the stopword list normally contains negations such as *not*, *no*, *nor* and *never*. We deliberately **keep them**, because removing "not" from "the phone is not great" would turn a negative text into a positive one. Stemming is left off (`USE_STEMMING = False`) to keep words readable; set it to `True` to apply NLTK's PorterStemmer.

## 4. Train / test split (80 / 20)

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["sentiment"])
print("Train:", train_df.shape[0], "| Test:", test_df.shape[0])
print(train_df["sentiment"].value_counts(normalize=True).round(3).to_dict())
print(test_df["sentiment"].value_counts(normalize=True).round(3).to_dict())

**Observation:** 2,508 texts for training and 627 for testing. Stratification keeps the class proportions almost identical in both sets (39.1% / 33.1% / 27.8%).

## 5. Feature extraction with TF-IDF

**What does TF-IDF measure?** *Term Frequency - Inverse Document Frequency* gives each word a weight that combines two ideas:
- **TF:** how often the word appears in this text (more = more important to this text);
- **IDF:** how rare the word is across all texts (words found everywhere, like "the", get a low weight; distinctive words, like "terrible" or "recommend", get a high weight).

So TF-IDF turns text into numbers that highlight the words that actually tell texts apart. We use **unigrams and bigrams** (`ngram_range=(1, 2)`) so that pairs like "not great" are captured as a feature, and `min_df=2` to ignore words that appear only once. The vectoriser is fitted on the training data only, to avoid leaking information from the test set.

In [ ]:
# Fit TF-IDF on the TRAINING data only (no information leaks from the test set)
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
X_train = tfidf.fit_transform(train_df["clean_text"])
X_test = tfidf.transform(test_df["clean_text"])
y_train, y_test = train_df["sentiment"], test_df["sentiment"]
print("TF-IDF matrix:", X_train.shape, "(texts x unique words/word pairs)")

## 6. Train the classifiers

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM (LinearSVC)": LinearSVC(),
}
preds, rows = {}, []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds[name] = model.predict(X_test)
    p, r, f, _ = precision_recall_fscore_support(y_test, preds[name], average="macro", zero_division=0)
    rows.append({"Model": name, "Accuracy": accuracy_score(y_test, preds[name]),
                 "Precision (macro)": p, "Recall (macro)": r, "F1 (macro)": f})

results = pd.DataFrame(rows).set_index("Model").round(3)
results

**Models:** (1) **Multinomial Naive Bayes** - a fast probabilistic baseline that assumes words are independent; (2) **Logistic Regression** - learns a weight for each word and bigram; (3) **Linear SVM** - finds the best separating boundary between classes, often very strong on text.

**Observation:** all three models score between **90.7% and 92.7% accuracy**. Linear SVM is best (accuracy 92.7%, macro-F1 0.928), Logistic Regression is close (92.3%, 0.924) and Naive Bayes is slightly behind (90.7%, 0.908).

## 7. Evaluation: accuracy, precision, recall, F1 and confusion matrices

In [ ]:
for name in models:
    print("=" * 60); print(name); print("=" * 60)
    print(classification_report(y_test, preds[name], digits=3))

labels = ["positive", "neutral", "negative"]
fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
for a, name in zip(ax, models):
    cm = confusion_matrix(y_test, preds[name], labels=labels)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, cbar=False, ax=a)
    a.set_title(f"{name}\nAccuracy = {accuracy_score(y_test, preds[name]):.3f}")
    a.set_xlabel("Predicted"); a.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("images/02_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

results.plot.bar(figsize=(10, 4.5), rot=0, ylim=(0.8, 1.0))
plt.title("Model Comparison (test set)"); plt.legend(loc="lower right")
plt.savefig("images/03_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

best_name = results["F1 (macro)"].idxmax()
print("Best model by macro-F1:", best_name)

**Observation:**
- The **positive** class has the highest recall (0.94-0.96) but the lowest precision (0.88-0.90): the models tend to **over-predict "positive"**, i.e. some neutral and negative texts get labelled positive.
- **Neutral** has the highest precision (0.95): when a model says "neutral" it is almost always right, but it misses about 7-11% of neutral texts.
- Logistic Regression and SVM beat Naive Bayes by about 1.6-2 points. Naive Bayes treats every word independently, whereas the other two learn from combinations of words (such as "not" + an adjective).
- The differences between the three models are small, so for this data the choice is more about simplicity and speed than accuracy.

## 8. WordClouds: most frequent words per sentiment

In [ ]:
try:
    from wordcloud import WordCloud
    HAVE_WC = True
except ImportError:          # in Colab: !pip install wordcloud
    HAVE_WC = False

top_words = {}
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for a, s, col in zip(ax, ["positive", "neutral", "negative"], ["Greens", "Greys", "Reds"]):
    freq = Counter(" ".join(df.loc[df["sentiment"] == s, "clean_text"]).split())
    top_words[s] = [w for w, _ in freq.most_common(10)]
    if HAVE_WC:
        wc = WordCloud(width=600, height=400, background_color="white", colormap=col).generate_from_frequencies(freq)
        a.imshow(wc, interpolation="bilinear"); a.axis("off")
    else:                    # fallback: top-15 word bar chart
        w, c = zip(*freq.most_common(15))
        a.barh(w[::-1], c[::-1], color=plt.get_cmap(col)(0.65))
    a.set_title(f"{s.capitalize()} texts")
plt.tight_layout()
plt.savefig("images/04_wordclouds.png", dpi=150, bbox_inches="tight")
plt.show()
print("WordCloud package available:", HAVE_WC)
pd.DataFrame(top_words, index=range(1, 11))

**Observation (top words from the table above):**
- **Positive:** *happy, recommend, love, highly, pleased* - clear emotional praise.
- **Negative:** *waste, money, keeps, failing, annoyed, hate* - complaints about value and reliability.
- **Neutral:** *bought, tried, works, arrived, received, Tuesday* - factual, delivery and usage words with no emotion.
- The word **"not"** is among the top words for both neutral and negative texts (e.g. "not great, not terrible" vs "not good"). A single word like "not" is therefore ambiguous, which is why using **bigrams** in TF-IDF helps.

(Where the `wordcloud` package is installed, each panel is drawn as a word cloud with larger words = more frequent; otherwise a top-word bar chart is shown.)

## 9. Error analysis: what does the best model get wrong?

In [ ]:
best_pred = preds[best_name]
errors = test_df.assign(predicted=best_pred)
errors = errors[errors["sentiment"] != errors["predicted"]]
print(f"{best_name}: {len(errors)} misclassified out of {len(test_df)} test texts ({len(errors)/len(test_df)*100:.1f}%)\n")

print("Most common mistakes (actual -> predicted):")
print(errors.groupby(["sentiment", "predicted"]).size().sort_values(ascending=False).head(4), "\n")

pattern = r"\bnot\b|\bbut\b|\bnever\b|\bno\b"
share_err = errors["text"].str.lower().str.contains(pattern).mean() * 100
share_all = test_df["text"].str.lower().str.contains(pattern).mean() * 100
print(f"Texts with 'not/but/never/no': {share_err:.0f}% of errors vs {share_all:.0f}% of all test texts\n")

pd.set_option("display.max_colwidth", 140)
errors.sample(5, random_state=1)[["text", "sentiment", "predicted"]].rename(columns={"sentiment": "actual"})

**Observation:** the best model (Linear SVM) misclassified **46 of 627 test texts (7.3%)**. The most common mistakes are *negative predicted as positive* (16), *positive predicted as negative* (10) and *neutral predicted as positive* (9).

**Why does it fail? (5 examples from this run)**
1. *"I really love the price, it is awesome. #loveit"* (labelled neutral, predicted positive): the text is clearly positive, so this is most likely a **wrong label** in the data. The model is arguably right.
2. *"HIGHLY RECOMMEND: THE WEBSITE IS OUTSTANDING, BUT I HATE THE PRICE, TOTALLY SLOW..."* (positive, predicted negative): a **mixed opinion**. The text contains both strong praise and strong complaints, and a bag-of-words model cannot tell which part matters most.
3. *"The store staff is great!, my brother has one too"* (labelled neutral, predicted positive): again clearly positive, so another **likely label error**.
4. *"So pleased, keyboard is awesome :), but the app is useless..."* (negative, predicted positive): **mixed opinion** where the final clause after "but" carries the real verdict; TF-IDF ignores word order, so it sees mostly positive words.
5. *"Very unhappy with the battery. horrible!..."* (labelled neutral, predicted negative): clearly negative, so a **label error** again.

**Patterns:** 61% of the model's errors contain *not / but / never / no*, versus only 26% of all test texts, so **negation and contrast ("but")** are the main weakness. In addition, some "errors" come from noisy labels rather than from the model. (Your sample of 5 errors may differ from these if you change the stopword list or random seed.)

## 10. Conclusion

- **Best model:** **Linear SVM** (accuracy 92.7%, macro-F1 0.928), narrowly ahead of Logistic Regression (92.3%) and Naive Bayes (90.7%). SVM or Logistic Regression would be the recommended choice; Naive Bayes is a good fast baseline.
- **Key challenges:** negation, mixed opinions ("great..., but...") and noisy labels. Word order and context are lost in TF-IDF; a model that reads word order (an LSTM or a transformer such as BERT) could improve on mixed-opinion cases.
- **Real-world application:** a company could run this model on tweets and product reviews to (a) automatically flag negative messages for the customer-support team, (b) track the share of positive vs negative opinion over time after a product launch, and (c) discover which topics (price, delivery, battery) drive complaints using the top-word analysis.
- **Limitation:** the results come from a synthetic, template-based dataset and are probably more optimistic than real social-media text, which is far more varied. On a real dataset, expect lower scores and re-check the observations.